### 各地でバラバラの土地利用情報のデータを1つの全国データとしてまとめる

In [9]:
!pip install geopandas pyogrio fiona shapely

In [10]:
from pathlib import Path
import zipfile
import pandas as pd
import geopandas as gpd
import gc
import numpy as np
import matplotlib.pyplot as plt

##### プロジェクトのパス設定

In [14]:
from pathlib import Path
import zipfile
import yaml

CONFIG_PATH = Path("config.yaml")

if CONFIG_PATH.exists():
    with open(CONFIG_PATH, "r", encoding="utf-8") as f:
        config = yaml.safe_load(f)
    DATA_DIR = Path(config["data_root"]).expanduser()
    if "project_root" in config:
        PROJECTS_DIR = Path(config["project_root"]).expanduser()
    else:
        PROJECTS_DIR = DATA_DIR.parent / "Projects"
else:
    DATA_DIR = Path("~/Desktop/Research/Data").expanduser()
    PROJECTS_DIR = Path("~/Desktop/Research/Projects").expanduser()

RESEARCH_DIR = DATA_DIR.parent

# ----------------------------
# LandUse
# ----------------------------
LANDUSE_RAW_DIR = DATA_DIR / "Raw" / "LandUse" / "2021" / "national_landuse"
LANDUSE_INTERMEDIATE_DIR = DATA_DIR / "Intermediate" / "LandUse" / "2021"
LANDUSE_EXTRACT_DIR = LANDUSE_INTERMEDIATE_DIR / "extracted" / "landuse_2021"
LANDUSE_PROCESSED_DIR = DATA_DIR / "Processed" / "LandUse" / "2021"

# ----------------------------
# Geospatial: ZIP polygon
# ----------------------------
ZIP_RAW_DIR = DATA_DIR / "Raw" / "Geospatial" / "ZIP_polygon" / "2022" / "2022_GEOSPACE郵便番号ポリゴン"
ZIP_SHAPE_DIR = ZIP_RAW_DIR / "shape"
ZIP_SHP_PATH = ZIP_SHAPE_DIR / "GSP99.shp"

# ----------------------------
# Geospatial: 500m mesh
# ----------------------------
MESH_RAW_DIR = DATA_DIR / "Raw" / "Geospatial" / "Mesh" / "500m_2024"
MESH_ZIP_PATH = MESH_RAW_DIR / "500m_mesh_2024_GEOJSON.zip"
MESH_EXTRACT_DIR = LANDUSE_INTERMEDIATE_DIR / "extracted" / "mesh_500m_2024"
MESH_GEOJSON_DIR = MESH_EXTRACT_DIR / "geojson"

# ----------------------------
# Project output
# ----------------------------
LANDUSE_BUILD_DIR = PROJECTS_DIR / "LandUse_build"
LANDUSE_MAP_DIR = LANDUSE_BUILD_DIR / "output" / "maps"

for p in [
    LANDUSE_EXTRACT_DIR,
    LANDUSE_PROCESSED_DIR,
    LANDUSE_INTERMEDIATE_DIR / "extracted" / "zipcode_polygon_2022",
    MESH_EXTRACT_DIR,
    MESH_GEOJSON_DIR,
    LANDUSE_MAP_DIR,
]:
    p.mkdir(parents=True, exist_ok=True)

print("DATA_DIR:", DATA_DIR)
print("PROJECTS_DIR:", PROJECTS_DIR)

RAW_ZIP_DIR = LANDUSE_RAW_DIR
EXTRACT_DIR = LANDUSE_EXTRACT_DIR
PROCESSED_DIR = LANDUSE_PROCESSED_DIR


LANDUSE_RAW_DIR: /Users/Tomo/Desktop/Research/Data/Raw/LandUse/2021/national_landuse
  exists: True

ZIP_SHP_PATH: /Users/Tomo/Desktop/Research/Data/Raw/Geospatial/ZIP_polygon/2022/2022_GEOSPACE郵便番号ポリゴン/shape/GSP99.shp
  exists: True

MESH_ZIP_PATH: /Users/Tomo/Desktop/Research/Data/Raw/Geospatial/Mesh/500m_2024/500m_mesh_2024_GEOJSON.zip
  exists: True


In [15]:
# 郵便番号ポリゴン zip を除外し、L03-b-21_ だけ対象にする
zip_files = sorted(RAW_ZIP_DIR.glob("L03-b-21_*.zip"))
print(f"land-use zip files found: {len(zip_files)}")

for z in zip_files[:5]:
    print(z.name)

for zip_path in zip_files:
    out_dir = EXTRACT_DIR / zip_path.stem
    out_dir.mkdir(parents=True, exist_ok=True)

    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(out_dir)

print("Unzip completed.")

land-use zip files found: 176
L03-b-21_3036-jgd2011_GML.zip
L03-b-21_3622-jgd2011_GML.zip
L03-b-21_3623-jgd2011_GML.zip
L03-b-21_3624-jgd2011_GML.zip
L03-b-21_3631-jgd2011_GML.zip
Unzip completed.


In [16]:
shp_files = sorted(EXTRACT_DIR.rglob("*.shp"))
print(f"shapefiles found: {len(shp_files)}")

for shp in shp_files[:5]:
    print(shp)

shapefiles found: 176
/Users/Tomo/Desktop/Research/Data/Intermediate/LandUse/2021/extracted/landuse_2021/L03-b-21_3036-jgd2011_GML/L03-b-21_3036.shp
/Users/Tomo/Desktop/Research/Data/Intermediate/LandUse/2021/extracted/landuse_2021/L03-b-21_3622-jgd2011_GML/L03-b-21_3622.shp
/Users/Tomo/Desktop/Research/Data/Intermediate/LandUse/2021/extracted/landuse_2021/L03-b-21_3623-jgd2011_GML/L03-b-21_3623.shp
/Users/Tomo/Desktop/Research/Data/Intermediate/LandUse/2021/extracted/landuse_2021/L03-b-21_3624-jgd2011_GML/L03-b-21_3624.shp
/Users/Tomo/Desktop/Research/Data/Intermediate/LandUse/2021/extracted/landuse_2021/L03-b-21_3631-jgd2011_GML/L03-b-21_3631.shp


In [17]:
gdfs = []
failed_files = []

for shp in shp_files:
    try:
        gdf = gpd.read_file(shp)

        # 元ファイル情報を残しておく
        gdf["source_file"] = shp.name
        gdf["source_folder"] = shp.parent.name

        gdfs.append(gdf)
        print(f"Read OK: {shp.name} | rows={len(gdf)}")

    except Exception as e:
        failed_files.append((str(shp), str(e)))
        print(f"Read FAILED: {shp.name} | {e}")

print(f"\nSuccess: {len(gdfs)} files")
print(f"Failed: {len(failed_files)} files")

# 失敗ファイル確認
if failed_files:
    failed_df = pd.DataFrame(failed_files, columns=["file", "error"])
    print(failed_df.head())

Read OK: L03-b-21_3036.shp | rows=20000
Read OK: L03-b-21_3622.shp | rows=10000
Read OK: L03-b-21_3623.shp | rows=110000
Read OK: L03-b-21_3624.shp | rows=130000
Read OK: L03-b-21_3631.shp | rows=10000
Read OK: L03-b-21_3641.shp | rows=10000
Read OK: L03-b-21_3653.shp | rows=10000
Read OK: L03-b-21_3724.shp | rows=20000
Read OK: L03-b-21_3725.shp | rows=110000
Read OK: L03-b-21_3741.shp | rows=30000
Read OK: L03-b-21_3823.shp | rows=50000
Read OK: L03-b-21_3824.shp | rows=10000
Read OK: L03-b-21_3831.shp | rows=50000
Read OK: L03-b-21_3841.shp | rows=10000
Read OK: L03-b-21_3926.shp | rows=60000
Read OK: L03-b-21_3927.shp | rows=340000
Read OK: L03-b-21_3928.shp | rows=70000
Read OK: L03-b-21_3942.shp | rows=30000
Read OK: L03-b-21_4027.shp | rows=70000
Read OK: L03-b-21_4028.shp | rows=110000
Read OK: L03-b-21_4040.shp | rows=40000
Read OK: L03-b-21_4042.shp | rows=60000
Read OK: L03-b-21_4128.shp | rows=90000
Read OK: L03-b-21_4129.shp | rows=30000
Read OK: L03-b-21_4142.shp | rows=5

In [18]:
crs_counts = pd.Series([str(gdf.crs) for gdf in gdfs]).value_counts()
print(crs_counts)

EPSG:6668    176
Name: count, dtype: int64


In [19]:
target_crs = gdfs[0].crs
gdfs_aligned = []

for gdf in gdfs:
    if gdf.crs != target_crs:
        gdf = gdf.to_crs(target_crs)
    gdfs_aligned.append(gdf)

print("CRS harmonized to:", target_crs)

CRS harmonized to: EPSG:6668


In [20]:
national_gdf = gpd.GeoDataFrame(
    pd.concat(gdfs_aligned, ignore_index=True),
    crs=target_crs
)

print(national_gdf.shape)
print(national_gdf.head())

(48900000, 6)
     L03b_001 L03b_002  L03b_003  \
0  3036400000     1500  20201014   
1  3036400001     1500  20201014   
2  3036400002     1500  20201014   
3  3036400003     1500  20201014   
4  3036400004     1500  20201014   

                                            geometry        source_file  \
0  POLYGON ((136 20.33333, 136 20.33417, 136.0012...  L03-b-21_3036.shp   
1  POLYGON ((136.00125 20.33333, 136.00125 20.334...  L03-b-21_3036.shp   
2  POLYGON ((136.0025 20.33333, 136.0025 20.33417...  L03-b-21_3036.shp   
3  POLYGON ((136.00375 20.33333, 136.00375 20.334...  L03-b-21_3036.shp   
4  POLYGON ((136.005 20.33333, 136.005 20.33417, ...  L03-b-21_3036.shp   

               source_folder  
0  L03-b-21_3036-jgd2011_GML  
1  L03-b-21_3036-jgd2011_GML  
2  L03-b-21_3036-jgd2011_GML  
3  L03-b-21_3036-jgd2011_GML  
4  L03-b-21_3036-jgd2011_GML  


In [21]:
out_gpkg = PROCESSED_DIR / "L03b21_national.gpkg"
national_gdf.to_file(out_gpkg, layer="landuse_mesh", driver="GPKG")

print("Saved:", out_gpkg)

Saved: /Users/Tomo/Desktop/Research/Data/Processed/LandUse/2021/L03b21_national.gpkg


### 各郵便番号ポリゴン内の土地利用用途別の面積を算出

In [1]:
import geopandas as gpd
import pandas as pd
from pathlib import Path
import zipfile
import gc
import matplotlib.pyplot as plt
import numpy as np

##### パス設定

In [3]:
from pathlib import Path
import zipfile
import yaml

CONFIG_PATH = Path("config.yaml")

if CONFIG_PATH.exists():
    with open(CONFIG_PATH, "r", encoding="utf-8") as f:
        config = yaml.safe_load(f)
    DATA_DIR = Path(config["data_root"]).expanduser()
    if "project_root" in config:
        PROJECTS_DIR = Path(config["project_root"]).expanduser()
    else:
        PROJECTS_DIR = DATA_DIR.parent / "Projects"
else:
    DATA_DIR = Path("~/Desktop/Research/Data").expanduser()
    PROJECTS_DIR = Path("~/Desktop/Research/Projects").expanduser()

RESEARCH_DIR = DATA_DIR.parent

# ----------------------------
# LandUse
# ----------------------------
LANDUSE_RAW_DIR = DATA_DIR / "Raw" / "LandUse" / "2021" / "national_landuse"
LANDUSE_INTERMEDIATE_DIR = DATA_DIR / "Intermediate" / "LandUse" / "2021"
LANDUSE_EXTRACT_DIR = LANDUSE_INTERMEDIATE_DIR / "extracted" / "landuse_2021"
LANDUSE_PROCESSED_DIR = DATA_DIR / "Processed" / "LandUse" / "2021"

# ----------------------------
# Geospatial: ZIP polygon
# ----------------------------
ZIP_RAW_DIR = DATA_DIR / "Raw" / "Geospatial" / "ZIP_polygon" / "2022" / "2022_GEOSPACE郵便番号ポリゴン"
ZIP_SHAPE_DIR = ZIP_RAW_DIR / "shape"
ZIP_SHP_PATH = ZIP_SHAPE_DIR / "GSP99.shp"

# ----------------------------
# Geospatial: 500m mesh
# ----------------------------
MESH_RAW_DIR = DATA_DIR / "Raw" / "Geospatial" / "Mesh" / "500m_2024"
MESH_ZIP_PATH = MESH_RAW_DIR / "500m_mesh_2024_GEOJSON.zip"
MESH_EXTRACT_DIR = LANDUSE_INTERMEDIATE_DIR / "extracted" / "mesh_500m_2024"
MESH_GEOJSON_DIR = MESH_EXTRACT_DIR / "geojson"

# ----------------------------
# Project output
# ----------------------------
LANDUSE_BUILD_DIR = PROJECTS_DIR / "LandUse_build"
LANDUSE_MAP_DIR = LANDUSE_BUILD_DIR / "output" / "maps"

for p in [
    LANDUSE_EXTRACT_DIR,
    LANDUSE_PROCESSED_DIR,
    LANDUSE_INTERMEDIATE_DIR / "extracted" / "zipcode_polygon_2022",
    MESH_EXTRACT_DIR,
    MESH_GEOJSON_DIR,
    LANDUSE_MAP_DIR,
]:
    p.mkdir(parents=True, exist_ok=True)

print("DATA_DIR:", DATA_DIR)
print("PROJECTS_DIR:", PROJECTS_DIR)

ZIP_EXTRACT_DIR = LANDUSE_INTERMEDIATE_DIR / "extracted" / "zipcode_polygon_2022"
RAW_DIR = ZIP_RAW_DIR
PROCESSED_DIR = LANDUSE_PROCESSED_DIR
landuse_path = PROCESSED_DIR / "L03b21_national.gpkg"
zipcode_shp_path = ZIP_SHP_PATH
zipcode_extract_dir = ZIP_EXTRACT_DIR


##### 郵便番号ポリゴンを解凍する

In [4]:
# すでに展開済みのshapefileを直接使う

shp_files = list(zipcode_shp_path.parent.glob("*.shp"))

if len(shp_files) == 0:
    raise FileNotFoundError("郵便番号ポリゴンの .shp が見つかりません。")

print("Found shapefile:", shp_files[0])

Found shapefile: /Users/Tomo/Desktop/Research/Data/Raw/Geospatial/ZIP_polygon/2022/2022_GEOSPACE郵便番号ポリゴン/shape/GSP99.shp


##### 郵便番号ポリゴンの読み込み

In [5]:
zip_gdf = gpd.read_file(shp_files[0])

zip_gdf = zip_gdf.rename(columns={
    "郵便番号": "zipcode",
    "市区町村CD": "citycode",
    "町域名": "town",
    "管理情報": "admin_info"
})

zip_gdf["zipcode"] = zip_gdf["zipcode"].astype(str).str.zfill(7)
zip_gdf["citycode"] = zip_gdf["citycode"].astype(str).str.zfill(5)

# 必要列だけ残す
zip_gdf = zip_gdf[["zipcode", "citycode", "geometry"]].copy()
# 都道府県コード（市区町村コードの先頭2桁）
zip_gdf["prefcode"] = zip_gdf["citycode"].astype(str).str[:2]

# 郵便番号が複数ポリゴンに分かれている場合に備えて dissolve
zip_gdf = zip_gdf[["zipcode", "prefcode", "geometry"]].copy()
zip_gdf = zip_gdf.dissolve(by=["zipcode", "prefcode"], as_index=False)

print(zip_gdf.head())
print(zip_gdf.crs)
print(zip_gdf.shape)

   zipcode prefcode                                           geometry
0  000None       01  MULTIPOLYGON (((146.15453 43.4114, 146.15456 4...
1  000None       23  POLYGON ((136.79324 35.25592, 136.79327 35.255...
2  0010010       01  POLYGON ((141.34908 43.07248, 141.34751 43.072...
3  0010011       01  POLYGON ((141.34879 43.07361, 141.34722 43.073...
4  0010012       01  POLYGON ((141.3485 43.07477, 141.34693 43.0745...
EPSG:6668
(113186, 3)


##### 居住地・可住地の算出のために必要な土地利用用途の設定+projection設定

In [6]:
target_codes = ["0500", "0700", "1100", "1400", "1500"]

# 全国一括の面積計算用：カスタム等積投影
JAPAN_AEA = (
    "+proj=aea +lat_1=30 +lat_2=46 +lat_0=36 +lon_0=138 "
    "+x_0=0 +y_0=0 +datum=WGS84 +units=m +no_defs"
)

zip_col = "zipcode"

##### 都道府県ごとに算出（一気にやると重いので）

In [7]:
results = []

pref_list = sorted(zip_gdf["prefcode"].dropna().unique())
print("pref count:", len(pref_list))

for pref in pref_list:
    print(f"\n=== Processing prefcode {pref} ===")

    # -----------------------------------------
    # 4-1. subset zipcode polygons
    # -----------------------------------------
    zip_sub = zip_gdf[zip_gdf["prefcode"] == pref].copy()
    if zip_sub.empty:
        print("zip_sub empty, skip")
        continue

    # 元CRSで bbox を作る（read_file の bbox に使う）
    xmin, ymin, xmax, ymax = zip_sub.total_bounds
    bbox = (xmin, ymin, xmax, ymax)

    # 全国版を全部メモリに載せない
    landuse_sub = gpd.read_file(landuse_path, bbox=bbox)

    if landuse_sub.empty:
        print("landuse_sub empty, skip")
        continue

    # 必要な土地利用コードだけ残す
    landuse_sub["L03b_002"] = landuse_sub["L03b_002"].astype(str).str.zfill(4)
    landuse_sub = landuse_sub[landuse_sub["L03b_002"].isin(target_codes)].copy()

    if landuse_sub.empty:
        print("No target landuse codes in this bbox, skip")
        continue

    # 必要列だけ
    landuse_sub = landuse_sub[["L03b_002", "geometry"]].copy()

    # 面積計算用に投影変換
    zip_sub = zip_sub[[zip_col, "geometry"]].copy().to_crs(JAPAN_AEA)
    landuse_sub = landuse_sub.to_crs(JAPAN_AEA)

    # 郵便番号総面積
    zip_sub["total_area_m2"] = zip_sub.geometry.area

    # 土地利用メッシュと郵便番号ポリゴンをオーバーレイ
    inter = gpd.overlay(
        landuse_sub,
        zip_sub[[zip_col, "total_area_m2", "geometry"]],
        how="intersection",
        keep_geom_type=False
    )

    if inter.empty:
        print("intersection empty, skip")
        continue

    inter["intersect_area_m2"] = inter.geometry.area

    # 面積0や極小を落とす
    inter = inter[inter["intersect_area_m2"] > 0].copy()

    # 郵便番号ごと・土地利用コードごとに集計
    area_by_code = (
        inter.groupby([zip_col, "L03b_002"], as_index=False)["intersect_area_m2"]
        .sum()
    )

    area_wide = (
        area_by_code
        .pivot(index=zip_col, columns="L03b_002", values="intersect_area_m2")
        .fillna(0)
        .reset_index()
    )
    area_wide.columns.name = None

    for c in target_codes:
        if c not in area_wide.columns:
            area_wide[c] = 0

    # わかりやすい列名を付ける
    area_wide = area_wide.rename(columns={
        "0500": "forest_area_m2",
        "0700": "building_area_m2",
        "1100": "water_area_m2",
        "1400": "beach_area_m2",
        "1500": "sea_area_m2"
    })

    total_area = zip_sub[[zip_col, "total_area_m2"]].copy()

    # 総面積と結合し、可住地面積・居住地面積を作る
    out = total_area.merge(area_wide, on=zip_col, how="left").fillna(0)

    out["habitable_area_m2"] = (
        out["total_area_m2"]
        - out["forest_area_m2"]
        - out["water_area_m2"]
        - out["beach_area_m2"]
        - out["sea_area_m2"]
    ).clip(lower=0)

    out["residential_area_m2"] = out["building_area_m2"].clip(lower=0)

    results.append(out)

    # メモリ整理
    del zip_sub, landuse_sub, inter, area_by_code, area_wide, total_area, out
    gc.collect()

pref count: 47

=== Processing prefcode 01 ===

=== Processing prefcode 02 ===

=== Processing prefcode 03 ===

=== Processing prefcode 04 ===

=== Processing prefcode 05 ===

=== Processing prefcode 06 ===

=== Processing prefcode 07 ===

=== Processing prefcode 08 ===

=== Processing prefcode 09 ===

=== Processing prefcode 10 ===

=== Processing prefcode 11 ===

=== Processing prefcode 12 ===

=== Processing prefcode 13 ===

=== Processing prefcode 14 ===

=== Processing prefcode 15 ===

=== Processing prefcode 16 ===

=== Processing prefcode 17 ===

=== Processing prefcode 18 ===

=== Processing prefcode 19 ===

=== Processing prefcode 20 ===

=== Processing prefcode 21 ===

=== Processing prefcode 22 ===

=== Processing prefcode 23 ===

=== Processing prefcode 24 ===

=== Processing prefcode 25 ===

=== Processing prefcode 26 ===

=== Processing prefcode 27 ===

=== Processing prefcode 28 ===

=== Processing prefcode 29 ===

=== Processing prefcode 30 ===

=== Processing prefcode 

##### 全データを集計

In [8]:
zipcode_area = pd.concat(results, ignore_index=True)

# 同じ郵便番号が重複した場合に備えて最後に再集計
zipcode_area = (
    zipcode_area.groupby("zipcode", as_index=False)
    .agg({
        "total_area_m2": "first",
        "forest_area_m2": "sum",
        "building_area_m2": "sum",
        "water_area_m2": "sum",
        "beach_area_m2": "sum",
        "sea_area_m2": "sum",
        "habitable_area_m2": "sum",
        "residential_area_m2": "sum"
    })
)

##### データの保存

In [9]:
# csvファイルとして保存
out_csv = PROCESSED_DIR / "zipcode_landuse_area.csv"
zipcode_area.to_csv(out_csv, index=False, encoding="utf-8-sig")
print("Saved:", out_csv)

# geodataとして保存（zip_gdf を再投影せず元のまま使って属性結合）
zipcode_out = zip_gdf.merge(zipcode_area, on="zipcode", how="left")

zipcode_out_path = PROCESSED_DIR / "zipcode_landuse_area.gpkg"
zipcode_out.to_file(zipcode_out_path, driver="GPKG")
print("Saved:", zipcode_out_path)

Saved: /Users/Tomo/Desktop/Research/Data/Processed/LandUse/2021/zipcode_landuse_area.csv
Saved: /Users/Tomo/Desktop/Research/Data/Processed/LandUse/2021/zipcode_landuse_area.gpkg


##### 作ったデータの内容確認

In [23]:
from pathlib import Path
import zipfile
import yaml

CONFIG_PATH = Path("config.yaml")

if CONFIG_PATH.exists():
    with open(CONFIG_PATH, "r", encoding="utf-8") as f:
        config = yaml.safe_load(f)
    DATA_DIR = Path(config["data_root"]).expanduser()
    if "project_root" in config:
        PROJECTS_DIR = Path(config["project_root"]).expanduser()
    else:
        PROJECTS_DIR = DATA_DIR.parent / "Projects"
else:
    DATA_DIR = Path("~/Desktop/Research/Data").expanduser()
    PROJECTS_DIR = Path("~/Desktop/Research/Projects").expanduser()

RESEARCH_DIR = DATA_DIR.parent

# ----------------------------
# LandUse
# ----------------------------
LANDUSE_RAW_DIR = DATA_DIR / "Raw" / "LandUse" / "2021" / "national_landuse"
LANDUSE_INTERMEDIATE_DIR = DATA_DIR / "Intermediate" / "LandUse" / "2021"
LANDUSE_EXTRACT_DIR = LANDUSE_INTERMEDIATE_DIR / "extracted" / "landuse_2021"
LANDUSE_PROCESSED_DIR = DATA_DIR / "Processed" / "LandUse" / "2021"

# ----------------------------
# Geospatial: ZIP polygon
# ----------------------------
ZIP_RAW_DIR = DATA_DIR / "Raw" / "Geospatial" / "ZIP_polygon" / "2022" / "2022_GEOSPACE郵便番号ポリゴン"
ZIP_SHAPE_DIR = ZIP_RAW_DIR / "shape"
ZIP_SHP_PATH = ZIP_SHAPE_DIR / "GSP99.shp"

# ----------------------------
# Geospatial: 500m mesh
# ----------------------------
MESH_RAW_DIR = DATA_DIR / "Raw" / "Geospatial" / "Mesh" / "500m_2024"
MESH_ZIP_PATH = MESH_RAW_DIR / "500m_mesh_2024_GEOJSON.zip"
MESH_EXTRACT_DIR = LANDUSE_INTERMEDIATE_DIR / "extracted" / "mesh_500m_2024"
MESH_GEOJSON_DIR = MESH_EXTRACT_DIR / "geojson"

# ----------------------------
# Project output
# ----------------------------
LANDUSE_BUILD_DIR = PROJECTS_DIR / "LandUse_build"
LANDUSE_MAP_DIR = LANDUSE_BUILD_DIR / "output" / "maps"

for p in [
    LANDUSE_EXTRACT_DIR,
    LANDUSE_PROCESSED_DIR,
    LANDUSE_INTERMEDIATE_DIR / "extracted" / "zipcode_polygon_2022",
    MESH_EXTRACT_DIR,
    MESH_GEOJSON_DIR,
    LANDUSE_MAP_DIR,
]:
    p.mkdir(parents=True, exist_ok=True)

print("DATA_DIR:", DATA_DIR)
print("PROJECTS_DIR:", PROJECTS_DIR)

ZIP_EXTRACT_DIR = LANDUSE_INTERMEDIATE_DIR / "extracted" / "zipcode_polygon_2022"
PROCESSED_DIR = LANDUSE_PROCESSED_DIR
MESH_BASE_DIR = MESH_GEOJSON_DIR
csv_path = PROCESSED_DIR / "zipcode_landuse_area.csv"
gpkg_path = PROCESSED_DIR / "zipcode_landuse_area.gpkg"


In [24]:
zipcode_area = pd.read_csv(csv_path, dtype={"zipcode": str})
zipcode_gdf = gpd.read_file(gpkg_path)

zipcode_area["zipcode"] = zipcode_area["zipcode"].astype(str).str.zfill(7)
zipcode_gdf["zipcode"] = zipcode_gdf["zipcode"].astype(str).str.zfill(7)


In [25]:
# =========================================================
# 3. Area consistency check / correction
# =========================================================
# 数値列を明示的に numeric にする
area_cols = [
    "total_area_m2",
    "forest_area_m2",
    "building_area_m2",
    "water_area_m2",
    "beach_area_m2",
    "sea_area_m2",
    "habitable_area_m2",
    "residential_area_m2"
]

for col in area_cols:
    zipcode_area[col] = pd.to_numeric(zipcode_area[col], errors="coerce")

# 負値をゼロに
zipcode_area[area_cols] = zipcode_area[area_cols].clip(lower=0)

# 物理的制約に合わせて上限補正
# 可住地面積 <= 総面積
zipcode_area["habitable_area_m2"] = zipcode_area[
    ["habitable_area_m2", "total_area_m2"]
].min(axis=1)

# 居住地面積 <= 可住地面積
zipcode_area["residential_area_m2"] = zipcode_area[
    ["residential_area_m2", "habitable_area_m2"]
].min(axis=1)

# 建物用地面積も参考として整合的にしておくならこれでもよい
zipcode_area["building_area_m2"] = zipcode_area[
    ["building_area_m2", "total_area_m2"]
].min(axis=1)

##### 割合列を追加

In [26]:
# =========================================================
# 4. Recalculate ratio columns
# =========================================================
# total_area_m2 == 0 の場合は NaN にする
denom = zipcode_area["total_area_m2"].replace(0, np.nan)

zipcode_area["habitable_ratio"] = (
    zipcode_area["habitable_area_m2"] / denom
).clip(lower=0, upper=1)

zipcode_area["residential_ratio"] = (
    zipcode_area["residential_area_m2"] / denom
).clip(lower=0, upper=1)

# 丸める
zipcode_area["habitable_ratio"] = zipcode_area["habitable_ratio"].round(6)
zipcode_area["residential_ratio"] = zipcode_area["residential_ratio"].round(6)


##### 保存

In [27]:
# =========================================================
# 6. Overwrite CSV
# =========================================================
zipcode_area.to_csv(csv_path, index=False, encoding="utf-8-sig")
print(f"Overwritten CSV: {csv_path}")

# =========================================================
# 7. Merge updated columns into GeoDataFrame
# =========================================================
# 既存の同名列を落とす
cols_to_drop = [
    "total_area_m2",
    "forest_area_m2",
    "building_area_m2",
    "water_area_m2",
    "beach_area_m2",
    "sea_area_m2",
    "habitable_area_m2",
    "residential_area_m2",
    "habitable_ratio",
    "residential_ratio"
]
zipcode_gdf = zipcode_gdf.drop(columns=cols_to_drop, errors="ignore")

zipcode_gdf = zipcode_gdf.merge(
    zipcode_area[
        [
            "zipcode",
            "total_area_m2",
            "forest_area_m2",
            "building_area_m2",
            "water_area_m2",
            "beach_area_m2",
            "sea_area_m2",
            "habitable_area_m2",
            "residential_area_m2",
            "habitable_ratio",
            "residential_ratio"
        ]
    ],
    on="zipcode",
    how="left"
)

# =========================================================
# 8. Overwrite GPKG
# =========================================================
zipcode_gdf.to_file(gpkg_path, driver="GPKG")
print(f"Overwritten GPKG: {gpkg_path}")

Overwritten CSV: /Users/Tomo/Desktop/Research/Data/Processed/LandUse/2021/zipcode_landuse_area.csv
Overwritten GPKG: /Users/Tomo/Desktop/Research/Data/Processed/LandUse/2021/zipcode_landuse_area.gpkg


#### データの整合性確認

##### 可住地割合

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(10, 12))

zipcode_gdf.plot(
    column="habitable_ratio",
    ax=ax,
    legend=True,
    cmap="viridis",
    linewidth=0
)

ax.set_title("Habitable Area Ratio by Zipcode")
ax.set_axis_off()
plt.show()

##### 居住地割合

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(10, 12))

zipcode_gdf.plot(
    column="residential_ratio",
    ax=ax,
    legend=True,
    cmap="viridis",
    linewidth=0
)

ax.set_title("Residential Area Ratio by Zipcode")
ax.set_axis_off()
plt.show()